# Whaleswap Module Guide

This notebook demonstrates end-to-end usage of the Whaleswap module:
- AMM pools (create/update/liquidity, exact-in and exact-out swaps)
- Orderbook offers (make/take) with SettlementMode (ESCROW/LIQUID)
- Auctions (open + redeem without active bid)
- Discovery queries and metrics

We’ll register `foo.dys` and `bar.dys` and mint those denoms for use in demos.

## Setup

We’ll use `alice`, `bob`, and `charlie`. We'll register two names `foo.dys` and `bar.dys` via commit–reveal and then mint coins under these names for Whaleswap operations.

In [ ]:
%%bash
dysond keys show -a alice

In [ ]:
# Addresses
[ALICE] = ! dysond keys show -a alice
[BOB]   = ! dysond keys show -a bob
[CHARLIE] = ! dysond keys show -a charlie
print("ALICE:", ALICE)
print("BOB  :", BOB)
print("CHARLIE:", CHARLIE)

# Helpers
import json
from IPython.core.magic import register_line_magic

@register_line_magic
def sh(line):
    ip = get_ipython()
    out = ip.getoutput(line)
    joined = '\n'.join(out)
    try:
        return json.loads(joined)
    except json.JSONDecodeError:
        print("Error parsing as json: ", joined)
        return joined

In [ ]:
# Robust shell helper: parse JSON even when extra lines precede it (e.g., gas estimate)
import json, re
from IPython.core.getipython import get_ipython
from IPython.core.magic import register_line_magic

@register_line_magic
def sh(line: str):
    ip = get_ipython()
    out_lines = ip.getoutput(line) if ip else []
    joined = "\n".join(out_lines)
    # Find first JSON object/array in the output (nbconvert may include log lines before JSON)
    m = re.search(r"(\{[\s\S]*|\[[\s\S]*)", joined)
    if m:
        snippet = m.group(1)
        try:
            return json.loads(snippet)
        except json.JSONDecodeError:
            pass
    print("Error parsing as json: ", joined)
    return joined


## Register `foo.dys` and `bar.dys` (commit–reveal)

We’ll commit with a random salt and a reasonable valuation, then reveal each name.

In [ ]:
# --- foo.dys commit-reveal ---
FOO_NAME = "foo.dys"
FOO_SALT = "this is random"

out = %sh dysond query nameservice compute-hash --name "$FOO_NAME" --salt "$FOO_SALT" --committer "$ALICE" -o json
foo_hash = out['hex_hash']
print("foo hash:", foo_hash)

foo_commit_tx = %sh  dysond tx nameservice commit --commitment "$foo_hash" --valuation "100udys" --from alice -y -o json | dysond query wait-tx -o json
assert foo_commit_tx['code'] == 0, foo_commit_tx['raw_log']
events = [e for e in foo_commit_tx['events'] if e['type'].startswith('dysonprotocol')]
print(json.dumps(events, indent=2))

foo_reveal_tx = %sh dysond tx nameservice reveal --name "$FOO_NAME" --salt "$FOO_SALT" --from alice -y -o json | dysond query wait-tx -o json
assert foo_reveal_tx['code'] == 0, foo_reveal_tx['raw_log']
events = [e for e in foo_reveal_tx['events'] if e['type'].startswith('dysonprotocol')]
print(json.dumps(events, indent=2))
print("Registered:", FOO_NAME)

In [ ]:
# --- bar.dys commit-reveal ---
BAR_NAME = "bar.dys"
BAR_SALT = "this is random"

out = %sh dysond query nameservice compute-hash --name "$BAR_NAME" --salt "$BAR_SALT" --committer "$ALICE" -o json
bar_hash = out['hex_hash']
print("bar hash:", bar_hash)

bar_commit_tx = %sh dysond tx nameservice commit --commitment "$bar_hash" --valuation "100udys" --from alice -y -o json | dysond query wait-tx -o json
assert bar_commit_tx['code'] == 0, bar_commit_tx['raw_log']
events = [e for e in bar_commit_tx['events'] if e['type'].startswith('dysonprotocol')]
print(json.dumps(events, indent=2))

bar_reveal_tx = %sh dysond tx nameservice reveal --name "$BAR_NAME" --salt "$BAR_SALT" --from alice -y -o json | dysond query wait-tx -o json
assert bar_reveal_tx['code'] == 0, bar_reveal_tx['raw_log']
events = [e for e in bar_reveal_tx['events'] if e['type'].startswith('dysonprotocol')]
print(json.dumps(events, indent=2))
print("Registered:", BAR_NAME)

## Mint coins for `foo.dys` and `bar.dys`

We’ll mint solid denoms under each name for liquidity and trading.

In [ ]:
from decimal import Decimal, ROUND_CEILING

params = %sh dysond query nameservice params -o json

fee_per = Decimal(params["params"].get("mint_fee_per_coin", "0"))
units = Decimal("1000000")
mint_fee = int((units * fee_per).to_integral_value(rounding=ROUND_CEILING))
print("mint_fee_per_coin:", str(fee_per), "computed mint_fee:", mint_fee)

foo_mint_tx = %sh dysond tx nameservice mint-coins --amount "1000000$FOO_NAME" --mint-fee {mint_fee}udys --from alice -y -o json | dysond query wait-tx -o json
assert foo_mint_tx['code'] == 0, foo_mint_tx['raw_log']
events = [e for e in foo_mint_tx['events'] if e['type'].startswith('dysonprotocol')]
print(json.dumps(events, indent=2))

bar_mint_tx = %sh dysond tx nameservice mint-coins --amount "1000000$BAR_NAME" --mint-fee {mint_fee}udys --from alice -y -o json | dysond query wait-tx -o json
assert bar_mint_tx['code'] == 0, bar_mint_tx['raw_log']
events = [e for e in bar_mint_tx['events'] if e['type'].startswith('dysonprotocol')]
print(json.dumps(events, indent=2))


## Create AMM pool (foo.dys / bar.dys)

We’ll seed a pool with initial reserves and a fee (e.g., 0.3%).

In [ ]:
# Create the pool with two repeated --coins flags; fee 0.003
create_pool_tx = %sh dysond tx whaleswap create-pool --coins "100000$FOO_NAME" --coins "100000$BAR_NAME" --fee-rate "0.003udys" --min-collateral-ratio "1.5" --from alice --gas auto -y -o json | dysond query wait-tx -o json
assert create_pool_tx['code'] == 0, create_pool_tx['raw_log']

# Resolve pool_id by pair
pools_by_pair  = %sh dysond query whaleswap pools-by-pair --base-denom "$FOO_NAME" --quote-denom "$BAR_NAME" -o json
assert len(pools_by_pair.get('pools', [])) > 0, "No pool found for pair"
POOL_ID = pools_by_pair['pools'][0]['pool_id']
print("POOL_ID:", POOL_ID)

# Inspect the pool
%sh dysond query whaleswap pool --pool-id "$POOL_ID" -o json

## Add and remove liquidity

Demonstrate adding more reserves and later removing some shares.

In [ ]:
# Add liquidity using the pool's canonical coin order (keep earlier setup)
pool_info = %sh dysond query whaleswap pool --pool-id "{POOL_ID}" -o json


DENOM0 = pool_info['pool']['coins'][0]['denom']
DENOM1 = pool_info['pool']['coins'][1]['denom']
SHARES = pool_info['pool']['shares_denom']

ADD1 = f"10000{DENOM0}"
ADD2 = f"10000{DENOM1}"

add_liq_tx = %sh dysond tx whaleswap add-liquidity --pool-id "{POOL_ID}" --amounts "{ADD1}" --amounts "{ADD2}" --from alice --gas auto -y -o json | dysond query wait-tx -o json

print("add-liquidity raw:")
assert add_liq_tx['code'] == 0, add_liq_tx['raw_log']
events = [e for e in add_liq_tx['events'] if e['type'].startswith('dysonprotocol')]
print(json.dumps(events, indent=2))

# Check Alice's shares before remove
alice_bal = %sh dysond query bank balance "{ALICE}" "{SHARES}" -o json
avail = int(alice_bal['balance']['amount'])
print("Alice shares before remove:", alice_bal)
assert avail > 0, "No shares available"

# Remove a small portion of liquidity (ensure >0 and small)
REMOVE_SHARES = avail // 10
rm_liq_tx = %sh dysond tx whaleswap remove-liquidity --pool-id "{POOL_ID}" --shares "{REMOVE_SHARES}" --from alice --gas auto -y -o json | dysond query wait-tx -o json
assert rm_liq_tx['code'] == 0, rm_liq_tx['raw_log']

events = [e for e in rm_liq_tx['events'] if e['type'].startswith('dysonprotocol')]
print("Removed shares:")
print(json.dumps(events, indent=2))

# Check Alice's shares after remove
alice_bal = %sh dysond query bank balance "{ALICE}" "{SHARES}" -o json
avail = int(alice_bal['balance']['amount'])
print("Alice shares after remove:", alice_bal)
assert avail > 0, "No shares available"

## Pool swaps (exact-in and exact-out)

We show:
- Exact-in: provide `swap_in` and cap with `--max-input`
- Exact-out: provide `swap_out`, module computes required input

In [ ]:
# Fund demo accounts for swaps and takes
# - Bob needs foo.dys and bar.dys (for pool swap and taking Alice's offer)

tx = sh(f'dysond tx bank send alice "{BOB}" "100000{FOO_NAME}" -y -o json | dysond query wait-tx -o json')
assert tx['code'] == 0, tx['raw_log']

tx = sh(f'dysond tx bank send alice "{BOB}" "100000{BAR_NAME}" -y -o json | dysond query wait-tx -o json')
assert tx['code'] == 0, tx['raw_log']

# (Optional) quick balance peek
print("Bob balances:")
! dysond query bank balances "$BOB" -o json | jq -M

In [ ]:
# Pool swap (exact-in): single op object + min-output safety; keep intermediate output
import json, shlex

max_in = f"500{FOO_NAME}"
op = json.dumps({"swap": {"pool_id": int(POOL_ID), "swap_in": {"denom": FOO_NAME, "amount": "500"}}})
op_q = shlex.quote(op)

swap_in_tx = %sh dysond tx whaleswap make-trade --from bob --max-input {max_in} --op {op_q} --min-output "1$BAR_NAME" -y -o json | dysond query wait-tx -o json
events = [e for e in swap_in_tx['events'] if e['type'].startswith('dysonprotocol')]

assert swap_in_tx['code'] == 0, swap_in_tx['raw_log']
print("Exact-in swap success")
print(json.dumps(events, indent=2))

In [ ]:
# Pool swap (exact-out): single op object + provide cap for inferred input denom; keep intermediate output
import json, shlex

op = json.dumps({"swap": {"pool_id": int(POOL_ID), "swap_out": {"denom": BAR_NAME, "amount": "250"}}})
op_q = shlex.quote(op)

swap_out_tx = %sh dysond tx whaleswap make-trade --from bob --max-input "100000$FOO_NAME" --op {op_q} -y -o json | dysond query wait-tx -o json
assert swap_out_tx['code'] == 0, swap_out_tx['raw_log']

events = [e for e in swap_out_tx['events'] if e['type'].startswith('dysonprotocol')]
print("Exact-out swap success")
print(json.dumps(events, indent=2))

## Update pool config (optional)

Adjust fee or set price bands.

In [ ]:
# Lower fee to 0.25% (example)
upd_tx = %sh dysond tx whaleswap update-pool-config --pool-id "$POOL_ID" --fee-rate "0.0025udys" --from alice -y -o json | dysond query wait-tx -o json
assert upd_tx['code'] == 0, upd_tx['raw_log']
print("Pool fee updated")

events = [e for e in upd_tx['events'] if e['type'].startswith('dysonprotocol')]
print("Pool fee updated")
print(json.dumps(events, indent=2))

# Re-check pool
! dysond query whaleswap pool --pool-id "$POOL_ID" -o json

## Orderbook offers (make + take)

- Maker posts an offer (have → want)
- Taker executes with full remaining (no `take_units`) or partial (`take_units`).

In [ ]:
# Alice makes an offer: she has foo.dys and wants bar.dys
# Autocli typically accepts coin syntax for Coin fields; adjust if your CLI differs
mk_offer_tx = %sh dysond tx whaleswap make-offer --have "1000$FOO_NAME" --want "400$BAR_NAME" --from alice -y -o json | dysond query wait-tx -o json
assert mk_offer_tx['code'] == 0, mk_offer_tx['raw_log']
print("Offer created")
events = [e for e in mk_offer_tx['events'] if e['type'].startswith('dysonprotocol')]
print(json.dumps(events, indent=2))

# Find the offer id by owner
offers_by_owner = %sh dysond query whaleswap offers-by-owner --owner "$ALICE" -o json
assert len(offers_by_owner.get('offers', [])) > 0, "No offers found for Alice"
OFFER_ID = int(offers_by_owner['offers'][0]['offer_id'])
print("OFFER_ID:", OFFER_ID)
print(json.dumps(offers_by_owner, indent=2))

# Bob takes the offer fully via make-trade
import shlex
take_op = json.dumps({"take": {"offer_id": OFFER_ID}})
take_op_q = shlex.quote(take_op)
take_tx = %sh dysond tx whaleswap make-trade --max-input "400$BAR_NAME" --op {take_op_q} --from bob -y -o json | dysond query wait-tx -o json
assert take_tx['code'] == 0, take_tx['raw_log']
print("Offer taken by Bob")
events = [e for e in take_tx['events'] if e['type'].startswith('dysonprotocol')]
print(json.dumps(events, indent=2))

## Liquid-mode offers (no wrappers)

Create an offer with SettlementMode LIQUID (no escrow; locks PFAND) and take it.

In [ ]:
# Alice makes a liquid-mode offer: have 100 foo.dys, want 90 bar.dys
mk = %sh dysond tx whaleswap make-offer --have "100$FOO_NAME" --want "90$BAR_NAME" --settlement-mode settlement-liquid --from alice -y -o json | dysond query wait-tx -o json
assert mk['code'] == 0, mk['raw_log']
offer_id = int([a['value'] for e in mk['events'] for a in e['attributes'] if a['key']=='offer_id'][0])

# Bob takes the offer fully via make-trade
import shlex
take_op = json.dumps({"take": {"offer_id": offer_id}})
take_op_q = shlex.quote(take_op)
tk = %sh dysond tx whaleswap make-trade --max-input "90$BAR_NAME" --op {take_op_q} --from bob -y -o json | dysond query wait-tx -o json
assert tk['code'] == 0, tk['raw_log']

## Auctions: open and redeem (no active bid)

- Seller escrows one solid coin in whaleswap and gets an NFT (class per bid denom).
- Without active bids, owner can redeem and burn the NFT.

In [ ]:
# Alice opens an auction: sell bar.dys, bid denom is foo.dys
open_auc_tx = %sh dysond tx whaleswap open-auction --sell "200$BAR_NAME" --bid-denom "$FOO_NAME" --from alice -y --gas 300000 -o json | dysond query wait-tx -o json
assert open_auc_tx['code'] == 0, open_auc_tx['raw_log']
print("Auction opened")
events = [e for e in open_auc_tx['events'] if e['type'].startswith('dysonprotocol')]
print(json.dumps(events, indent=2))

# Resolve auction id by seller
aucs_by_seller = %sh dysond query whaleswap auctions-by-seller --seller "$ALICE" -o json
assert len(aucs_by_seller.get('auctions', [])) > 0, "No auctions found"
print(json.dumps(aucs_by_seller, indent=2))
AUC_ID = aucs_by_seller['auctions'][-1]['auction_id']
print("AUC_ID:", AUC_ID)

# No active bid; owner redeems escrow
redeem_tx = %sh dysond tx whaleswap redeem-auction --auction-id "$AUC_ID" --from alice -y -o json | dysond query wait-tx -o json
assert redeem_tx['code'] == 0, redeem_tx['raw_log']
print("Auction redeemed and closed")
events = [e for e in redeem_tx['events'] if e['type'].startswith('dysonprotocol')]
print(json.dumps(events, indent=2))


## Mixed Trade (MakeTrade) with Note

Demonstrate `MsgMakeTrade` which combines pool swaps and orderbook takes in one transaction, with an optional note recorded on each resulting Trade.

We'll use a single swap leg for simplicity, including a note.



In [ ]:
# Mixed trade: single swap leg via MakeTrade, with a note
# Note: MsgMakeTrade requires JSON for operations; here a swap leg

import json, shlex

# Define a swap operation: exact-in 300 foo.dys for bar.dys output
# The autocli --op flag expects either {"swap": {...}} or {"take": {...}}
op = {
    "swap": {
        "pool_id": int(POOL_ID),
        "swap_in": {"denom": FOO_NAME, "amount": "300"}
    }
}
op_json = json.dumps(op)
op_q = shlex.quote(op_json)

note = "Demo mixed trade with note"

make_trade_tx = %sh dysond tx whaleswap make-trade --from bob --max-input "100000$FOO_NAME" --op {op_q} --min-output "1$BAR_NAME" --trade-note "{note}" -y -o json | dysond query wait-tx -o json
assert isinstance(make_trade_tx, dict) and make_trade_tx['code'] == 0, make_trade_tx

print("MakeTrade success with note")
events = [e for e in make_trade_tx['events'] if e['type'].startswith('dysonprotocol')]
print(json.dumps(events, indent=2))

# Query recent trade to verify note (look for EventTradeRecorded and extract trade_id)
trade_events = [e for e in events if 'EventTradeRecorded' in e['type']]
if trade_events:
    # Find trade_id attribute
    for attr in trade_events[0]['attributes']:
        if attr['key'] == 'trade_id':
            recent_trade_id = int(attr['value'].strip('"'))
            trade = %sh dysond query whaleswap trade --trade-id {recent_trade_id} -o json
            print("Recent trade with note:")
            print(json.dumps(trade, indent=2))
            break
else:
    print("No EventTradeRecorded found")


## Discovery queries and metrics

Explore indexes and health metrics.

In [ ]:
# Pools by denom
print("Pools mentioning foo.dys")
! dysond query whaleswap pools-by-denom --denom "$FOO_NAME" -o json | jq -M

# Offers by denom (role unspecified: either side)
print("Offers mentioning bar.dys (if any now):")
! dysond query whaleswap offers-by-denom --denom "$BAR_NAME" -o json | jq -M

# Auctions by pair price range (placeholder scanning endpoint)
print("Auctions (pair price range) foo.dys/bar.dys:")
! dysond query whaleswap auctions-by-pair-price-range --sell-denom "$BAR_NAME" --bid-denom "$FOO_NAME" -o json | jq -M

# Metrics
print("Module metrics:")
! dysond query whaleswap metrics -o json | jq -M

## Summary

- Registered `foo.dys` and `bar.dys`, minted denoms, and created an AMM pool with fee control and liquidity ops.
- Performed pool swaps (exact-in and exact-out).
- Demonstrated orderbook make/take using coins with SettlementMode (ESCROW/LIQUID).
- Opened and redeemed an auction without active bids.
- Ran discovery queries and fetched module metrics.

Tip: If any CLI flag formats differ in your environment (autocli vs custom CLI), run `dysond tx whaleswap --help` or the subcommand `--help` to confirm accepted flags and adjust accordingly.